# AICE Associate 대비 실습 교재
## Chapter 01. 데이터 로딩과 결합 (Data Loading & Merging)

본 교재는 **AICE Associate** 자격증 시험을 대비하여 데이터 분석의 첫 걸음인 **데이터 파일 불러오기**와 **데이터프레임 결합(concat, merge, join)**을 직접 코드 실습으로 다룹니다.

---

### 📋 학습 목차
1. **Section 01. 데이터 파일 불러오기** (CSV, Excel, JSON)
2. **Section 02. 데이터프레임 결합하기** (`pd.concat()`, `pd.merge()`, `DataFrame.join()`)
3. **Section 03. AICE 시험 실전 유형 연습** (JSON + CSV → Key 기준 Inner Merge)

## 00. 실습 환경 설정 및 샘플 데이터 생성
실습에 필요한 Python 패키지를 임포트하고, 로딩 및 결합 실습용 데이터 파일(CSV, Excel, JSON)을 생성합니다.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [21]:
'''
set root directory according your environment
'''
ROOT_DIR = '/content/drive/MyDrive/Colab Notebooks/AS_260824'

In [22]:
import os
import pandas as pd
import numpy as np
import json

# 1. 샘플 CSV 파일 생성 (고객 정보)
df_csv = pd.DataFrame({
    'RID': ['R001', 'R002', 'R003', 'R004', 'R005'],
    'Name': ['김민수', '이서연', '박지훈', '최수진', '정태양'],
    'Age': [28, 34, 22, 45, 31],
    'City': ['서울', '부산', '대구', '인천', '광주']
})
df_csv.to_csv(os.path.join(ROOT_DIR, 'sample_customers.csv'), index=False, encoding='utf-8-sig')

# 2. 샘플 Excel 파일 생성 (추가 고객 정보)
df_excel = pd.DataFrame({
    'RID': ['R006', 'R007', 'R008'],
    'Name': ['강하늘', '윤아름', '한동훈'],
    'Age': [29, 38, 41],
    'City': ['대전', '울산', '수원']
})
df_excel.to_excel(os.path.join(ROOT_DIR, 'sample_customers_extra.xlsx'), index=False)

# 3. 샘플 JSON 파일 생성 (구매/성적 정보)
df_json = pd.DataFrame({
    'RID': ['R001', 'R002', 'R003', 'R006', 'R009'],
    'Score': [85, 92, 76, 88, 95],
    'Grade': ['B', 'A', 'C', 'B', 'A']
})
df_json.to_json(os.path.join(ROOT_DIR, 'sample_scores.json'), orient='records', force_ascii=False)

print('✅ 실습용 데이터 파일 생성 완료: sample_customers.csv, sample_customers_extra.xlsx, sample_scores.json')

✅ 실습용 데이터 파일 생성 완료: sample_customers.csv, sample_customers_extra.xlsx, sample_scores.json


---
## Section 01. 데이터 파일 불러오기

AICE Associate 시험에서는 다양한 형태의 파일(CSV, Excel, JSON 등)을 Pandas DataFrame으로 올바르게 로딩하는 능력을 평가합니다.

| 파일 형식 | 로딩 함수 | 저장 함수 |
|---|---|---|
| CSV | `pd.read_csv('파일명.csv')` | `df.to_csv()` |
| Excel | `pd.read_excel('파일명.xlsx')` | `df.to_excel()` |
| JSON | `pd.read_json('파일명.json')` | `df.to_json()` |

In [7]:
# 1. CSV 파일 읽기
df_cust = pd.read_csv(os.path.join(ROOT_DIR, 'sample_customers.csv'))
print('--- [CSV 로딩 결과] ---')
display(df_cust)

# 2. Excel 파일 읽기
df_cust_extra = pd.read_excel(os.path.join(ROOT_DIR, 'sample_customers_extra.xlsx'))
print('\n--- [Excel 로딩 결과] ---')
display(df_cust_extra)

# 3. JSON 파일 읽기
df_score = pd.read_json(os.path.join(ROOT_DIR, 'sample_scores.json'))
print('\n--- [JSON 로딩 결과] ---')
display(df_score)

--- [CSV 로딩 결과] ---


,RID,Name,Age,City
0,R001,김민수,28,서울
1,R002,이서연,34,부산
2,R003,박지훈,22,대구
3,R004,최수진,45,인천
4,R005,정태양,31,광주



--- [Excel 로딩 결과] ---


,RID,Name,Age,City
0,R006,강하늘,29,대전
1,R007,윤아름,38,울산
2,R008,한동훈,41,수원



--- [JSON 로딩 결과] ---


,RID,Score,Grade
0,R001,85,B
1,R002,92,A
2,R003,76,C
3,R006,88,B
4,R009,95,A


---
## Section 02. 데이터프레임 결합하기

Pandas에서 데이터를 결합하는 방법은 목적에 따라 **3가지**로 구분됩니다.

| 방법 | 핵심 개념 | 기준 | 시험 중요도 |
|---|---|---|---|
| `pd.concat()` | 데이터를 단순 연결 (위/아래 또는 좌/우) | `axis=0` (행) / `axis=1` (열) | ★★★☆☆ |
| `pd.merge()` | 공통 Key를 기준으로 결합 | Key 컬럼 (`on='컬럼명'`) | ★★★★★ |
| `DataFrame.join()` | Index를 기준으로 결합 | Index | ★★☆☆☆ |


---
### 1. pd.merge() — 공통 Key 기준 결합

공통으로 존재하는 컬럼(Key)을 기준으로 결합하는 대표적인 방법입니다.

#### Merge의 4가지 방식 (`how` 파라미터)
- **`inner`** (기본값): 양쪽 모두에 존재하는 Key만 유지 (교집합)
- **`left`**: 왼쪽 DataFrame의 모든 Key 유지 + 오른쪽 데이터 결합
- **`right`**: 오른쪽 DataFrame의 모든 Key 유지 + 왼쪽 데이터 결합
- **`outer`**: 양쪽 모든 Key 유지 (합집합)

In [10]:
# 예시용 간단 데이터
df_left = pd.DataFrame({'ID': [1, 2, 3], '이름': ['김민수', '이서연', '박지훈']})
df_right = pd.DataFrame({'ID': [1, 2, 4], '점수': [85, 92, 78]})

display(df_left)
display(df_right)

,ID,이름
0,1,김민수
1,2,이서연
2,3,박지훈


,ID,점수
0,1,85
1,2,92
2,4,78


In [11]:
print('=== [INNER MERGE] (둘 다 있는 것) ===')
display(pd.merge(df_left, df_right, on='ID', how='inner'))

print('\n=== [LEFT MERGE] (왼쪽 기준) ===')
display(pd.merge(df_left, df_right, on='ID', how='left'))

print('\n=== [RIGHT MERGE] (오른쪽 기준) ===')
display(pd.merge(df_left, df_right, on='ID', how='right'))

print('\n=== [OUTER MERGE] (모두) ===')
display(pd.merge(df_left, df_right, on='ID', how='outer'))

=== [INNER MERGE] (둘 다 있는 것) ===


,ID,이름,점수
0,1,김민수,85
1,2,이서연,92



=== [LEFT MERGE] (왼쪽 기준) ===


,ID,이름,점수
0,1,김민수,85.0
1,2,이서연,92.0
2,3,박지훈,NaN



=== [RIGHT MERGE] (오른쪽 기준) ===


,ID,이름,점수
0,1,김민수,85
1,2,이서연,92
2,4,NaN,78



=== [OUTER MERGE] (모두) ===


,ID,이름,점수
0,1,김민수,85.0
1,2,이서연,92.0
2,3,박지훈,NaN
3,4,NaN,78.0


### 2. pd.concat() — 단순 연결 (위아래 / 좌우)
- `axis=0` (기본값): 세로 방향으로 행을 아래에 붙입니다.
- `axis=1`: 가로 방향으로 열을 옆에 붙입니다.
- `ignore_index=True`: 기존 인덱스를 재정렬합니다.

In [13]:
# 1. CSV 파일 읽기
print('--- [CSV 로딩 결과] ---')
display(df_cust)

# 2. Excel 파일 읽기
#df_cust_extra = pd.read_excel(os.path.join(ROOT_DIR, 'sample_customers_extra.xlsx'))
print('\n--- [Excel 로딩 결과] ---')
display(df_cust_extra)

# 3. JSON 파일 읽기
#df_score = pd.read_json(os.path.join(ROOT_DIR, 'sample_scores.json'))
print('\n--- [JSON 로딩 결과] ---')
display(df_score)

--- [CSV 로딩 결과] ---


,RID,Name,Age,City
0,R001,김민수,28,서울
1,R002,이서연,34,부산
2,R003,박지훈,22,대구
3,R004,최수진,45,인천
4,R005,정태양,31,광주



--- [Excel 로딩 결과] ---


,RID,Name,Age,City
0,R006,강하늘,29,대전
1,R007,윤아름,38,울산
2,R008,한동훈,41,수원



--- [JSON 로딩 결과] ---


,RID,Score,Grade
0,R001,85,B
1,R002,92,A
2,R003,76,C
3,R006,88,B
4,R009,95,A


In [14]:
# 세로 방향 결합 (axis=0)
df_concat_v = pd.concat([df_cust, df_cust_extra], axis=0, ignore_index=True)
print('--- [concat 세로 결합 결과] ---')
display(df_concat_v)

--- [concat 세로 결합 결과] ---


,RID,Name,Age,City
0,R001,김민수,28,서울
1,R002,이서연,34,부산
2,R003,박지훈,22,대구
3,R004,최수진,45,인천
4,R005,정태양,31,광주
5,R006,강하늘,29,대전
6,R007,윤아름,38,울산
7,R008,한동훈,41,수원


In [16]:
# 가로 방향 결합 (axis=1)
df_cust.iloc[:3, :2]
df_concat_h = pd.concat([df_cust.iloc[:3, :2], df_score.iloc[:3, 1:]], axis=1)
print('\n--- [concat 가로 결합 결과] ---')
display(df_concat_h)


--- [concat 가로 결합 결과] ---


,RID,Name,Score,Grade
0,R001,김민수,85,B
1,R002,이서연,92,A
2,R003,박지훈,76,C


In [15]:
display(df_cust.iloc[:3, :2])
display(df_score.iloc[:3, 1:])

,RID,Name
0,R001,김민수
1,R002,이서연
2,R003,박지훈


,Score,Grade
0,85,B
1,92,A
2,76,C


---
### 3. DataFrame.join() — Index 기준 결합

DataFrame의 **인덱스(Index)**를 기준으로 데이터를 결합할 때 사용합니다.

In [17]:
df_idx1 = pd.DataFrame({'A': ['A0', 'A1', 'A2']}, index=['i1', 'i2', 'i3'])
df_idx2 = pd.DataFrame({'B': ['B0', 'B1', 'B2']}, index=['i1', 'i2', 'i4'])

# Index 기준 join
df_joined = df_idx1.join(df_idx2, how='inner')
print('--- [join() 결합 결과 (inner)] ---')
print(df_joined)

--- [join() 결합 결과 (inner)] ---
     A   B
i1  A0  B0
i2  A1  B1


---
## Section 03. AICE 시험 실전 유형 연습

AICE Associate 실기 시험에 자주 출제되는 **"JSON + CSV 파일 로딩 후 공통 Key(RID) 기준 Inner Merge"** 실습 문제입니다.

### 🎯 실전 문제
1. `sample_customers.csv` 파일과 `sample_scores.json` 파일을 로딩하세요.
2. 두 데이터프레임을 공통 식별자인 `RID` 컬럼을 기준으로 `inner` 조건 결합을 수행하세요.
3. 결합된 결과를 `df_merged`에 저장하고, 데이터 크기(`shape`), 정보(`info()`), 상위 5개 행(`head()`)을 확인하세요.

In [18]:
# [실전 풀이]
# Step 1. 파일 불러오기
#df_csv_data = pd.read_csv('sample_customers.csv')
#df_json_data = pd.read_json('sample_scores.json')
df_csv_data = df_cust
df_json_data = df_score



In [19]:
# Step 2. 공통 Key('RID') 기준 Inner Merge
df_merged = pd.merge(df_csv_data, df_json_data, on='RID', how='inner')

# Step 3. 결과 확인
print('1. 데이터 크기 (shape):', df_merged.shape)
print('\n2. 데이터 정보 (info):')
df_merged.info()

print('\n3. 상위 데이터 (head):')
display(df_merged.head())

1. 데이터 크기 (shape): (3, 6)

2. 데이터 정보 (info):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   RID     3 non-null      object
 1   Name    3 non-null      object
 2   Age     3 non-null      int64 
 3   City    3 non-null      object
 4   Score   3 non-null      int64 
 5   Grade   3 non-null      object
dtypes: int64(2), object(4)
memory usage: 276.0+ bytes

3. 상위 데이터 (head):


,RID,Name,Age,City,Score,Grade
0,R001,김민수,28,서울,85,B
1,R002,이서연,34,부산,92,A
2,R003,박지훈,22,대구,76,C
